# Análisis de correlación del mercado inmobiliario argentino

## 1. Introducción

En este proyecto se analizarán las relaciones entre distintas variables del mercado inmobiliario argentino a partir de un conjunto de datos de publicaciones de propiedades.

Se utilizarán los métodos de correlación de Pearson, Spearman y Kendall para observar cómo se relacionan variables como el precio, la superficie, la cantidad de ambientes y la antigüedad de las propiedades.

También se analizará cómo la distribución de los datos y la presencia de valores atípicos pueden influir en los resultados de cada método..

### Objetivos

- Analizar las relaciones entre las principales variables numéricas del mercado inmobiliario.

- Calcular y comparar los coeficientes de correlación de Pearson, Spearman y Kendall.

- Observar cómo se distribuyen los datos y detectar posibles valores atípicos.

- Identificar qué método se adapta mejor a los datos analizados.

- Interpretar los resultados y tener presente que una correlación no implica causalidad.


## 2. Importación de bibliotecas

In [ ]:
# Importación de bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import normaltest

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 3. Comprobación de la ruta del archivo

In [ ]:
# Verificar la ubicación del archivo original

ruta = "../data/venta.csv"

print("¿Existe el archivo?", os.path.exists(ruta))
print("Ruta absoluta:", os.path.abspath(ruta))

## 4. Carga del dataset

In [ ]:
# Carga del dataset original

df = pd.read_csv(
    ruta,
    sep="|",
    encoding="latin-1",
    low_memory=False
)

print("Dimensiones del dataset:", df.shape)


## 5. Revisión inicial del dataset

In [ ]:
#Primeros registros e informe general
display(df.head())
df.info()

La revisión inicial permitió observar que el conjunto de datos contiene 35 columnas con información sobre las propiedades, como precios, superficies, cantidad de ambientes, antigüedad, ubicación y fecha de publicación.

También se identificaron valores faltantes y algunos datos que será necesario revisar durante la etapa de exploración y limpieza. En esta instancia no se realizaron modificaciones, ya que el objetivo fue conocer la estructura y el contenido general del dataset.

## 6. Selección de variables

In [59]:

# Variables seleccionadas para el análisis de correlación

columnas_analisis = [
    "id",
    "p_dolar",
    "sup_tot",
    "sup_cub",
    "amb",
    "antig",
    "barrio"
]

df_analisis = df[columnas_analisis].copy()

print("Registros:", df_analisis.shape[0])
print("Dimensiones:", df_analisis.shape)

display(df_analisis.head())

Registros: 519792
Dimensiones: (519792, 7)


,id,p_dolar,sup_tot,sup_cub,amb,antig,barrio
0,42781433,580000.00,500.0,265.0,4.0,35.0,NaN
1,41311197,309000.00,147.0,132.0,4.0,35.0,villa urquiza
2,41656002,238369.31,81.0,81.0,3.0,0.0,nuñez
3,42642945,147695.00,52.0,46.0,2.0,NaN,saavedra
4,42080435,87000.00,36.0,31.0,1.0,NaN,caballito


Se seleccionaron las variables necesarias para realizar el análisis de correlación. Se incluyeron el precio en dólares, la superficie total y cubierta, la cantidad de ambientes y la antigüedad de las propiedades.

También se conservaron las variables `id` y `barrio`. La primera permite identificar cada publicación y la segunda aporta información sobre su ubicación.

El conjunto seleccionado contiene 519.792 registros y 7 columnas. Para calcular las correlaciones se utilizarán únicamente las variables numéricas de interés.

## 7. Revisión de valores faltantes

In [60]:
# Cantidad y porcentaje de valores faltantes por columna

faltantes = pd.DataFrame({
    "Valores faltantes": df_analisis.isnull().sum(),
    "Porcentaje (%)": (df_analisis.isnull().mean() * 100).round(2)
})

# Ordenar de mayor a menor porcentaje
faltantes = faltantes.sort_values(
    by="Porcentaje (%)",
    ascending=False
)

display(faltantes)

,Valores faltantes,Porcentaje (%)
antig,130286,25.07
sup_cub,26475,5.09
p_dolar,12354,2.38
sup_tot,7330,1.41
amb,6059,1.17
barrio,74,0.01
id,0,0.00


Se observa que la variable antigüedad ('antig') presenta la mayor cantidad de valores faltantes, con un 25,07 % de los registros. En segundo lugar se encuentra la superficie cubierta ('sup_cub'), con un 5,09 %.

Las variables precio en dólares, superficie total y cantidad de ambientes tienen menos del 3 % de valores faltantes.

Estos datos se tendrán en cuenta durante la limpieza y el análisis, ya que la cantidad de registros disponibles puede variar según las variables que se comparen.

## 8. Análisis exploratorio de datos (EDA)

En esta sección se analizarán las principales variables del dataset para conocer sus distribuciones, identificar posibles relaciones entre ellas y detectar valores atípicos o inconsistencias que puedan influir en los resultados de los análisis de correlación.

El análisis exploratorio buscará responder las siguientes preguntas:

¿Cómo se distribuyen los precios y las superficies de las propiedades?

¿Existen valores extremos que puedan influir en los resultados de las correlaciones?

¿Qué relación se observa entre el precio y las características de las propiedades, como la superficie, la cantidad de ambientes y la antigüedad?

¿Existen inconsistencias en los datos que deban considerarse antes de realizar los cálculos?

¿Qué características de las variables pueden ayudar a determinar si es más apropiado utilizar Pearson, Spearman o Kendall?

### 8.1. Estadísticas descriptivas iniciales

In [62]:

# 8.1. Estadísticas descriptivas de las variables numéricas

variables_numericas = [
    "p_dolar",
    "sup_tot",
    "sup_cub",
    "amb",
    "antig"
]

display(
    df_analisis[variables_numericas]
    .describe()
    .round(2)
)


,p_dolar,sup_tot,sup_cub,amb,antig
count,5.074380e+05,512462.00,493317.00,513733.00,389506.00
mean,1.870911e+06,102.28,81.15,2.72,26.05
std,4.133081e+08,1381.74,511.19,1.61,43.42
min,1.555200e+02,1.00,0.00,0.00,-9.00
25%,1.190000e+05,44.00,40.00,2.00,1.00
50%,1.700000e+05,65.00,58.00,3.00,25.00
75%,2.850000e+05,103.00,90.00,4.00,45.00
max,1.111111e+11,382637.00,130000.00,543.00,2019.00


Las estadísticas descriptivas muestran diferencias importantes entre la media y la mediana de algunas variables, especialmente en el precio y las superficies. Esto indica que existen valores muy alejados de los valores habituales.

También se observan datos que necesitan una revisión, como propiedades con una cantidad muy elevada de ambientes, superficies muy grandes y antigüedades negativas o inusuales.

Por estos motivos, será necesario analizar con más detalle la distribución de las variables y revisar los posibles valores atípicos antes de calcular las correlaciones.

### 8.2. Análisis de superficies

In [63]:

# Estadísticas descriptivas de las superficies

display(
    df_analisis[["sup_tot", "sup_cub"]].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2)
)



,sup_tot,sup_cub
count,512462.00,493317.00
mean,102.28,81.15
std,1381.74,511.19
min,1.00,0.00
25%,44.00,40.00
50%,65.00,58.00
75%,103.00,90.00
90%,170.00,148.00
95%,224.00,198.00
99%,386.00,332.00


In [64]:

# Revisar las 15 superficies totales más grandes

display(
    df_analisis.sort_values(
        "sup_tot", ascending=False
    )[
        ["id", "p_dolar", "amb", "sup_tot",
         "sup_cub", "barrio"]
    ].head(15)
)


,id,p_dolar,amb,sup_tot,sup_cub,barrio
505613,45780358,600000.0,5.0,382637.0,170.00,liniers
127826,43236529,900000.0,5.0,263960.0,263.96,belgrano
139423,43236529,900000.0,5.0,263960.0,263.96,belgrano
115763,43236529,905000.0,5.0,263960.0,263.96,belgrano
81150,43236529,920000.0,5.0,263960.0,263.96,belgrano
151845,43236529,900000.0,5.0,263960.0,263.96,belgrano
132133,43236529,900000.0,5.0,263960.0,263.96,belgrano
101957,43236529,905000.0,5.0,263960.0,263.96,belgrano
396647,44874318,290000.0,3.0,255760.0,199.00,villa del parque
68019,43178986,330000.0,5.0,178800.0,178.80,flores


In [65]:

# Revisar las 15 superficies cubiertas más grandes

display(
    df_analisis.sort_values(
        "sup_cub", ascending=False
    )[
        ["id", "p_dolar", "amb", "sup_tot",
         "sup_cub", "barrio"]
    ].head(15)
)


,id,p_dolar,amb,sup_tot,sup_cub,barrio
363159,44713363,305000.0,3.0,137000.0,130000.0,palermo
350126,44713363,305000.0,3.0,137000.0,130000.0,palermo
422232,44977517,220000.0,5.0,126062.0,126062.0,caballito
396725,44360238,215000.0,5.0,126062.0,126062.0,caballito
409187,44360238,220000.0,5.0,126062.0,126062.0,caballito
514480,45896950,290000.0,3.0,125600.0,114560.0,barrio norte
518868,45885614,240000.0,4.0,104890.0,104890.0,barrio norte
518867,45885613,240000.0,4.0,104890.0,104890.0,barrio norte
474626,45485421,170000.0,2.0,66050.0,46250.0,nuñez
499888,45704670,138000.0,2.0,42000.0,42000.0,belgrano


In [66]:

# Revisar superficies iguales a cero

print(
    "Superficie total igual a cero:",
    (df_analisis["sup_tot"] == 0).sum()
)

print(
    "Superficie cubierta igual a cero:",
    (df_analisis["sup_cub"] == 0).sum()
)


Superficie total igual a cero: 0
Superficie cubierta igual a cero: 105


In [67]:

# Cantidad de superficies superiores a 1.000 m²

print(
    "Superficie total mayor a 1.000 m²:",
    (df_analisis["sup_tot"] > 1000).sum()
)

print(
    "Superficie cubierta mayor a 1.000 m²:",
    (df_analisis["sup_cub"] > 1000).sum()
)


Superficie total mayor a 1.000 m²: 491
Superficie cubierta mayor a 1.000 m²: 160


In [68]:

# Identificar inconsistencias entre superficie cubierta y total

inconsistencias_sup = df_analisis[
    (df_analisis["sup_cub"] > df_analisis["sup_tot"]) &
    df_analisis["sup_cub"].notna() &
    df_analisis["sup_tot"].notna()
].copy()

# Calcular la diferencia entre ambas superficies
inconsistencias_sup["diferencia_sup"] = (
    inconsistencias_sup["sup_cub"] -
    inconsistencias_sup["sup_tot"]
)

print("Registros con superficie cubierta mayor que la total:",
      len(inconsistencias_sup))

display(
    inconsistencias_sup[
        ["id", "p_dolar", "amb", "sup_tot",
         "sup_cub", "diferencia_sup", "barrio"]
    ].head(15)
)


Registros con superficie cubierta mayor que la total: 1569


,id,p_dolar,amb,sup_tot,sup_cub,diferencia_sup,barrio
45,42457408,1.530000e+05,1.0,33.000,60.0,27.000,palermo
321,42518309,2.200000e+05,3.0,64.000,69.0,5.000,palermo hollywood
589,42454717,1.900000e+05,3.0,66.000,69.0,3.000,belgrano
634,42742391,8.500000e+04,2.0,34.000,36.0,2.000,centro
637,41017485,1.500000e+06,5.0,5.008,508.0,502.992,belgrano
890,42811966,8.500000e+04,2.0,42.000,49.0,7.000,congreso
1448,42743283,8.500000e+04,2.0,40.000,46.0,6.000,coghlan
1527,42743910,1.000000e+05,1.0,38.000,41.0,3.000,caballito
1777,42840623,4.829545e+03,3.0,46.000,53.0,7.000,congreso
1835,42796131,1.190000e+05,2.0,56.000,57.0,1.000,caballito


In [69]:

# Clasificar las diferencias entre superficies por rangos

rangos_diferencia = pd.cut(
    inconsistencias_sup["diferencia_sup"],
    bins=[0, 5, 20, 50, 100, float("inf")],
    labels=[
        "1 a 5 m²",
        "Más de 5 a 20 m²",
        "Más de 20 a 50 m²",
        "Más de 50 a 100 m²",
        "Más de 100 m²"
    ]
)

display(
    rangos_diferencia.value_counts().sort_index()
)


diferencia_sup
1 a 5 m²              606
Más de 5 a 20 m²      267
Más de 20 a 50 m²     376
Más de 50 a 100 m²    241
Más de 100 m²          79
Name: count, dtype: int64

El análisis de las superficies muestra que la mayoría de las propiedades tiene dimensiones dentro de valores habituales, pero también aparecen publicaciones con superficies muy elevadas.

Además, se encontraron 1.569 registros donde la superficie cubierta es mayor que la superficie total. En algunos casos las diferencias son pequeñas, mientras que en otros son mucho más importantes.

Estos resultados muestran que existen datos que necesitan una revisión más detallada. Por el momento, se conservarán los registros para analizarlos durante la etapa de limpieza y evaluación de valores atípicos.

### 8.3. Análisis de antigüedad y ambientes

### 8.3.1. Análisis de antigüedad.

In [70]:
# Revisar posibles valores anómalos en antigüedad y ambientes

print("Antigüedad negativa:")
print((df_analisis["antig"] < 0).sum())

print("\nAmbientes iguales a cero:")
print((df_analisis["amb"] == 0).sum())

print("\nAmbientes mayores a 10:")
print((df_analisis["amb"] > 10).sum())



Antigüedad negativa:
38336

Ambientes iguales a cero:
2354

Ambientes mayores a 10:
410


Se encontraron algunos valores que necesitan una revisión en las variables antigüedad y cantidad de ambientes.

Hay 38.336 registros con antigüedad negativa, 2.354 propiedades con cero ambientes y 410 publicaciones que informan más de 10 ambientes.

Estos casos se revisarán con mayor detalle para decidir qué tratamiento aplicar durante la limpieza de los datos.

In [71]:

# Revisar los valores más frecuentes de antigüedad

display(
    df_analisis["antig"]
    .value_counts(dropna=False)
    .sort_index()
    .head(25)
)


antig
-9.0     21616
-2.0        20
-1.0     16700
 0.0     55521
 1.0      7008
 2.0      6944
 3.0      5461
 4.0      5865
 5.0      9774
 6.0      5501
 7.0      4899
 8.0      7278
 9.0      3306
 10.0    12477
 11.0     1543
 12.0     3032
 13.0     1221
 14.0     1357
 15.0     6107
 16.0     1210
 17.0     1275
 18.0     2491
 19.0     1169
 20.0     9503
 21.0      475
Name: count, dtype: int64

Al revisar la distribución de la variable antigüedad, se encontraron 38.336 registros con valores negativos, principalmente -9 y -1, y en menor medida -2. Estos valores podrían corresponder a códigos utilizados en la fuente original o a errores en los datos.

También se observa una cantidad importante de propiedades con antigüedad igual a cero (55.521 registros), que podrían representar inmuebles nuevos o a estrenar.

Por este motivo, se revisará el significado de estos valores antes de definir cómo tratarlos. No se eliminarán automáticamente, ya que podrían contener información válida.

In [72]:
# Revisar los valores negativos más frecuentes de antigüedad

display(
    df_analisis.loc[
        df_analisis["antig"] < 0, "antig"
    ].value_counts().head(15)
)

antig
-9.0    21616
-1.0    16700
-2.0       20
Name: count, dtype: int64

Se confirma que los valores negativos se concentran principalmente en -9 y -1. Esta información se tendrá en cuenta al definir el tratamiento de la variable antigüedad.

In [73]:
# Clasificar valores extremos de antigüedad por intervalos

rangos_antig = {
    "Entre 101 y 999": (
        (df_analisis["antig"] > 100) &
        (df_analisis["antig"] < 1000)
    ),
    "Entre 1000 y 1899": (
        (df_analisis["antig"] >= 1000) &
        (df_analisis["antig"] < 1900)
    ),
    "Entre 1900 y 2026": (
        df_analisis["antig"].between(1900, 2026)
    ),
    "Mayores a 2026": (
        df_analisis["antig"] > 2026
    )
}

for nombre, condicion in rangos_antig.items():
    print(f"{nombre}: {condicion.sum():,} registros")


Entre 101 y 999: 933 registros
Entre 1000 y 1899: 1 registros
Entre 1900 y 2026: 104 registros
Mayores a 2026: 0 registros


Al revisar los valores más altos de antigüedad, se encontraron 933 registros entre 101 y 999 años, uno entre 1000 y 1899 años y 104 registros entre 1900 y 2026.

Algunos de estos valores podrían corresponder a años de construcción ingresados en lugar de la antigüedad del inmueble. Sin embargo, como no se puede confirmar su significado, será necesario revisarlos antes de decidir cómo tratarlos.

In [79]:
# Distribución de valores entre 1900 y 2026

display(
    df_analisis.loc[
        df_analisis["antig"].between(1900, 2026),
        "antig"
    ].value_counts().sort_index().to_frame("Cantidad de registros")
)

,Cantidad de registros
antig,
1928.0,3
1930.0,3
1936.0,7
1940.0,5
1945.0,1
1946.0,3
1950.0,7
1959.0,1
1960.0,3


In [80]:
# Distribución de posibles años de construcción por barrio

antig_anios = df_analisis[
    df_analisis["antig"].between(1900, 2026)
]

display(
    antig_anios["barrio"]
    .value_counts(dropna=False)
    .to_frame("Cantidad de registros")
)


,Cantidad de registros
barrio,
recoleta,22
balvanera,8
villa urquiza,8
constitucion,7
coghlan,6
belgrano,5
palermo,5
almagro,5
congreso,4


Al agrupar estos registros por barrio, se observa cómo se distribuyen las publicaciones que presentan valores de antigüedad entre 1900 y 2026.

Esta revisión permite conocer en qué barrios aparecen estos casos, aunque no permite confirmar que los valores correspondan efectivamente a años de construcción.

### 8.3.2. Análisis de ambientes

In [75]:
# Mostrar las 10 propiedades con mayor superficie total

display(
    df_analisis.nlargest(10, "sup_tot")[
        ["id", "p_dolar", "amb", "antig",
         "sup_tot", "sup_cub", "barrio"]
    ]
)



,id,p_dolar,amb,antig,sup_tot,sup_cub,barrio
505613,45780358,600000.0,5.0,90.0,382637.0,170.00,liniers
81150,43236529,920000.0,5.0,35.0,263960.0,263.96,belgrano
101957,43236529,905000.0,5.0,35.0,263960.0,263.96,belgrano
115763,43236529,905000.0,5.0,35.0,263960.0,263.96,belgrano
127826,43236529,900000.0,5.0,35.0,263960.0,263.96,belgrano
132133,43236529,900000.0,5.0,35.0,263960.0,263.96,belgrano
139423,43236529,900000.0,5.0,35.0,263960.0,263.96,belgrano
151845,43236529,900000.0,5.0,35.0,263960.0,263.96,belgrano
396647,44874318,290000.0,3.0,49.0,255760.0,199.00,villa del parque
68019,43178986,330000.0,5.0,40.0,178800.0,178.80,flores


In [78]:

# Distribución de ambientes superiores a 10

display(
    df_analisis.loc[df_analisis["amb"] > 10, "amb"]
    .value_counts()
    .sort_index()
)

# Revisar los registros con más de 10 ambientes
amb_extremos = df_analisis[df_analisis["amb"] > 10]

display(
    amb_extremos[
        ["id", "amb", "p_dolar", "antig",
         "sup_tot", "sup_cub", "barrio"]
    ].sort_values("amb", ascending=False).head(20)
)


amb
11.0      99
12.0     100
13.0      50
14.0      20
15.0      26
16.0       4
17.0       9
18.0       4
19.0       1
20.0      12
21.0       3
22.0       6
24.0       4
28.0       4
30.0      16
31.0       1
32.0       5
34.0      10
40.0      11
42.0       1
43.0       1
44.0       4
45.0       2
50.0       2
51.0       1
52.0       4
54.0       1
56.0       4
60.0       2
107.0      2
543.0      1
Name: count, dtype: int64

,id,amb,p_dolar,antig,sup_tot,sup_cub,barrio
55667,43145587,543.0,8.000000e+05,4.0,281.0,212.0,villa del parque
135553,43502998,107.0,3.300000e+05,-1.0,107.0,107.0,caballito
145757,43574367,107.0,3.300000e+05,-1.0,107.0,100.0,caballito
273468,44204170,60.0,1.520000e+05,60.0,72.0,70.0,congreso
259041,44204170,60.0,1.520000e+05,60.0,72.0,70.0,congreso
90855,43290168,56.0,1.050000e+05,NaN,58.0,56.0,san telmo
69430,42740100,56.0,1.050000e+06,NaN,248.0,198.0,palermo chico
96993,43290168,56.0,1.050000e+05,NaN,58.0,56.0,san telmo
127985,43290168,56.0,1.050000e+05,NaN,58.0,56.0,san telmo
444288,45171864,54.0,1.250000e+05,36.0,88.0,61.0,caballito


Al revisar las 10 propiedades con mayor superficie total, se observan valores muy elevados en comparación con las dimensiones habituales de las propiedades del conjunto de datos.

También se identifican diferencias importantes entre la superficie total y la superficie cubierta en algunos registros. Por ejemplo, aparecen propiedades con superficies totales de varios miles de metros cuadrados.

Estos casos podrían corresponder a propiedades de características especiales o a datos cargados de manera incorrecta. Por este motivo, se revisarán junto con el resto de los valores extremos, pero no se eliminarán automáticamente.

**Interpretación de valores extremos de antigüedad**

Al ampliar la revisión de la variable `antig`, se identificaron 1.038 registros con valores superiores a 100 años.

De ellos, 933 presentan valores entre 101 y 999 años, uno registra un valor entre 1000 y 1899 años y 104 corresponden al intervalo entre 1900 y 2026.

Los valores comprendidos entre 1900 y 2026 podrían representar años de construcción cargados en lugar de la antigüedad expresada en años. Por otra parte, el registro con un valor entre 1000 y 1899 resulta especialmente llamativo y requiere una revisión individual.

No se encontraron valores superiores a 2026.

Estos resultados evidencian posibles inconsistencias en la variable `antig`, que deberán considerarse durante la limpieza y al evaluar su influencia en los coeficientes de correlación.

## 9. Limpieza y tratamiento de datos

En esta sección se tratarán los valores inconsistentes identificados durante el análisis exploratorio. Se conservarán los registros originales y se reemplazarán por valores faltantes únicamente aquellos datos que no resulten válidos o confiables.

### 9.1. Tratamiento de valores de antigüedad

In [81]:
df_analisis = df[columnas_analisis].copy()

In [82]:
print("Valores negativos:", (df_analisis["antig"] < 0).sum())
print("Valores entre 1900 y 2019:",
      df_analisis["antig"].between(1900, 2019).sum())
print("Valores superiores a 100:",
      (df_analisis["antig"] > 100).sum())
print("Valores faltantes:", df_analisis["antig"].isna().sum())

Valores negativos: 38336
Valores entre 1900 y 2019: 104
Valores superiores a 100: 1038
Valores faltantes: 130286


In [83]:
# Limpieza de la variable antigüedad

# Crear una copia de trabajo
df_analisis = df_analisis.copy()

# Contabilizar los valores antes de modificarlos
negativos = (df_analisis["antig"] < 0).sum()

anios_sospechosos = (
    df_analisis["antig"].between(1900, 2019)
).sum()

antig_mayor_100 = (
    df_analisis["antig"] > 100
).sum()

# Tratar valores negativos como faltantes
df_analisis.loc[
    df_analisis["antig"] < 0, "antig"
] = np.nan

# Tratar posibles años de construcción mal cargados
df_analisis.loc[
    df_analisis["antig"].between(1900, 2019),
    "antig"
] = np.nan

# Mostrar resultados
print("Valores negativos tratados:", negativos)
print("Posibles años tratados:", anios_sospechosos)
print("Valores originales superiores a 100:", antig_mayor_100)

print(
    "Valores superiores a 100 restantes:",
    (df_analisis["antig"] > 100).sum()
)

print(
    "Total de valores faltantes:",
    df_analisis["antig"].isna().sum()
)

display(df_analisis["antig"].describe().round(2))

Valores negativos tratados: 38336
Posibles años tratados: 104
Valores originales superiores a 100: 1038
Valores superiores a 100 restantes: 934
Total de valores faltantes: 168726


count    351066.00
mean         28.91
std          29.03
min           0.00
25%           5.00
50%          30.00
75%          45.00
max        1030.00
Name: antig, dtype: float64

Se identificaron valores negativos y registros comprendidos entre 1900 y 2019 en la variable antigüedad. Estos últimos podrían corresponder a años de construcción ingresados en lugar de la antigüedad del inmueble.

Como no fue posible confirmar su significado, ambos grupos se reemplazaron por valores faltantes (NaN).

Los valores superiores a 100 años que no pertenecían al intervalo tratado se conservaron para una revisión posterior, evitando eliminarlos sin evidencia suficiente.

In [84]:
# Distribución de antigüedades superiores a 100 años

display(
    df_analisis.loc[
        df_analisis["antig"] > 100,
        "antig"
    ].value_counts().sort_index()
)

antig
101.0     20
102.0     16
103.0      7
104.0     19
105.0     69
          ..
990.0      8
991.0      1
993.0      2
998.0      1
1030.0     1
Name: count, Length: 84, dtype: int64

Luego del tratamiento, se conservaron 934 registros con antigüedades superiores a 100 años. Al revisar su distribución, se observa que existen valores muy variados, algunos de ellos extremadamente elevados.

Como no se cuenta con información suficiente para confirmar si son errores o datos válidos, se decidió mantenerlos en el conjunto de datos para no eliminarlos sin justificación.

### 9.1.2. Inspección de registros extremos

In [85]:
# Inspeccionar registros con antigüedad superior a 100 años

antig_extremos = df_analisis[
    df_analisis["antig"] > 100
]

display(
    antig_extremos[
        ["id", "antig", "p_dolar", "amb",
         "sup_tot", "sup_cub", "barrio"]
    ].sort_values(
        "antig",
        ascending=False
    ).head(20)
)

,id,antig,p_dolar,amb,sup_tot,sup_cub,barrio
90770,43311611,1030.0,1350000.0,NaN,300.0,300.0,recoleta
291014,44339486,998.0,355000.0,3.0,131.0,90.0,nuñez
388193,44642721,993.0,315000.0,4.0,131.0,116.0,flores
348603,44642721,993.0,360000.0,4.0,131.0,116.0,flores
316575,44522683,991.0,139000.0,2.0,51.0,45.0,caballito
259769,44192063,990.0,160000.0,2.0,60.0,56.0,parque chacabuco
277289,44148048,990.0,390000.0,4.0,93.0,83.0,belgrano
217943,44009242,990.0,410000.0,4.0,93.0,83.0,belgrano
245255,44148048,990.0,410000.0,4.0,93.0,83.0,belgrano
374540,44762057,990.0,440000.0,NaN,202.0,202.0,saavedra


In [86]:
# Revisar los 20 valores de antigüedad más frecuentes
display(
    antig_extremos["antig"]
    .value_counts()
    .head(20)
)

antig
110.0    130
120.0     90
105.0     69
119.0     68
121.0     43
109.0     33
107.0     30
108.0     27
115.0     25
101.0     20
104.0     19
200.0     19
118.0     18
102.0     16
970.0     16
116.0     15
400.0     15
114.0     15
130.0     14
170.0     13
Name: count, dtype: int64

In [87]:
# Distribución de antigüedades superiores a 100 por barrio

display(
    antig_extremos["barrio"]
    .value_counts()
    .head(15)
)

barrio
monserrat        145
balvanera         98
san telmo         88
recoleta          72
centro            71
congreso          48
belgrano          43
barrio norte      32
san nicolas       30
almagro           28
villa crespo      24
palermo           21
palermo chico     19
caballito         17
san cristobal     17
Name: count, dtype: int64

Al revisar los registros con antigüedad superior a 100 años, se observa que algunos presentan valores extremadamente elevados, como 970, 990 y hasta 1030 años.

También se encontraron valores que se repiten con mayor frecuencia, como 110 y 120 años. Por otra parte, los registros aparecen en distintos barrios, principalmente Monserrat, Balvanera, San Telmo y Recoleta.

Estos resultados muestran que existen antigüedades poco habituales, pero no permiten confirmar si se trata de errores de carga o de datos válidos. Por este motivo, se decidió conservarlos para el análisis, sin eliminarlos automáticamente.

### 9.1.3. Tratamiento de valores de ambientes

In [88]:

# Reemplazar cero ambientes por valores faltantes

amb_cero = (df_analisis["amb"] == 0).sum()

df_analisis.loc[df_analisis["amb"] == 0, "amb"] = np.nan

print("Valores cero tratados:", amb_cero)
print("Valores cero restantes:", (df_analisis["amb"] == 0).sum())
print("Valores faltantes actuales:", df_analisis["amb"].isna().sum())


Valores cero tratados: 2354
Valores cero restantes: 0
Valores faltantes actuales: 8413


In [89]:
# Revisar propiedades con más de 50 ambientes en los datos originales

display(
    df.loc[
        df["amb"] > 50,
        ["id", "amb", "p_dolar", "sup_tot",
         "sup_cub", "barrio"]
    ].sort_values("amb", ascending=False)
)

,id,amb,p_dolar,sup_tot,sup_cub,barrio
55667,43145587,543.0,8.000000e+05,281.0,212.0,villa del parque
135553,43502998,107.0,3.300000e+05,107.0,107.0,caballito
145757,43574367,107.0,3.300000e+05,107.0,100.0,caballito
273468,44204170,60.0,1.520000e+05,72.0,70.0,congreso
259041,44204170,60.0,1.520000e+05,72.0,70.0,congreso
90855,43290168,56.0,1.050000e+05,58.0,56.0,san telmo
69430,42740100,56.0,1.050000e+06,248.0,198.0,palermo chico
96993,43290168,56.0,1.050000e+05,58.0,56.0,san telmo
127985,43290168,56.0,1.050000e+05,58.0,56.0,san telmo
444288,45171864,54.0,1.250000e+05,88.0,61.0,caballito


In [ ]:


# Reemplazar valores superiores a 50 ambientes por valores faltantes

amb_extremos = (df_analisis["amb"] > 50).sum()

df_analisis.loc[df_analisis["amb"] > 50, "amb"] = np.nan

print("Valores extremos tratados:", amb_extremos)
print("Valores superiores a 50 restantes:", (df_analisis["amb"] > 50).sum())
print("Valores faltantes actuales:", df_analisis["amb"].isna().sum())

Valores extremos tratados: 15
Valores superiores a 50 restantes: 0
Valores faltantes actuales: 8428


In [91]:
# Distribución de ambientes después del tratamiento

display(
    df_analisis["amb"]
    .value_counts(dropna=False)
    .sort_index()
)

amb
1.0      98896
2.0     136914
3.0     143316
4.0      92427
5.0      26645
6.0       7901
7.0       2878
8.0       1357
9.0        347
10.0       288
11.0        99
12.0       100
13.0        50
14.0        20
15.0        26
16.0         4
17.0         9
18.0         4
19.0         1
20.0        12
21.0         3
22.0         6
24.0         4
28.0         4
30.0        16
31.0         1
32.0         5
34.0        10
40.0        11
42.0         1
43.0         1
44.0         4
45.0         2
50.0         2
NaN       8428
Name: count, dtype: int64

In [92]:
# Revisar propiedades con más de 50 ambientes en los datos originales

display(
    df.loc[
        df["amb"] > 50,
        ["id", "amb", "p_dolar", "sup_tot",
         "sup_cub", "barrio"]
    ].sort_values("amb", ascending=False)
)

,id,amb,p_dolar,sup_tot,sup_cub,barrio
55667,43145587,543.0,8.000000e+05,281.0,212.0,villa del parque
135553,43502998,107.0,3.300000e+05,107.0,107.0,caballito
145757,43574367,107.0,3.300000e+05,107.0,100.0,caballito
273468,44204170,60.0,1.520000e+05,72.0,70.0,congreso
259041,44204170,60.0,1.520000e+05,72.0,70.0,congreso
90855,43290168,56.0,1.050000e+05,58.0,56.0,san telmo
69430,42740100,56.0,1.050000e+06,248.0,198.0,palermo chico
96993,43290168,56.0,1.050000e+05,58.0,56.0,san telmo
127985,43290168,56.0,1.050000e+05,58.0,56.0,san telmo
444288,45171864,54.0,1.250000e+05,88.0,61.0,caballito


Durante la limpieza de la variable ambientes, se encontraron 2.354 registros con valor cero, que fueron reemplazados por valores faltantes (NaN), ya que no representan una cantidad válida de ambientes.

También se revisaron las propiedades con más de 50 ambientes en el conjunto original. Se encontraron 15 registros con valores extremos, algunos de ellos muy alejados de las cantidades habituales y que no guardaban una relación clara con las superficies informadas.

Por este motivo, esos 15 valores también se reemplazaron por valores faltantes. Después del tratamiento, la mayor parte de las publicaciones se concentra entre 1 y 5 ambientes, mientras que los valores superiores a 50 ya no aparecen en la variable.

## 10. Medidas resumen y análisis de distribución

En esta sección se analizará la distribución de las principales variables numéricas del conjunto de datos mediante las siguientes medidas estadísticas:

Media y mediana.

Desvío estándar.

Cuartiles y percentiles.

Asimetría y curtosis.

El objetivo es conocer la tendencia central, la dispersión y la forma de las distribuciones, identificando posibles asimetrías y valores extremos que puedan influir en el análisis de correlación.

### 10.1. Selección de variables numéricas

In [ ]:
# Variables numéricas para el análisis de distribución

variables_numericas = [
    "p_dolar",
    "sup_tot",
    "sup_cub",
    "amb",
    "antig"
]

print("Variables seleccionadas:", variables_numericas)
print("Cantidad de registros:", len(df_analisis))

### 10.2. Estadísticas descriptivas

In [ ]:
# Medidas resumen de las variables numéricas

display(
    df_analisis[variables_numericas]
    .describe()
    .round(2)
)

Las medidas resumen muestran diferencias importantes entre las variables analizadas. En el precio en dólares y las superficies, la media es considerablemente superior a la mediana, lo que sugiere una distribución asimétrica y una posible influencia de valores extremos.

En la variable ambientes, la media es de 2,73 y la mediana de 3, con una mayor concentración entre 2 y 4 ambientes. En antigüedad, la media es de 28,91 años y la mediana de 30 años. Se conserva un valor máximo de 1.030 años, que fue inspeccionado previamente y se mantuvo por no contar con evidencia suficiente para modificarlo.

### 10.3. Asimetría y curtosis

In [ ]:
# Análisis de asimetría y curtosis

distribucion = pd.DataFrame({
    "Asimetría": df_analisis[variables_numericas].skew(),
    "Curtosis": df_analisis[variables_numericas].kurt()
})

display(distribucion.round(2))

El análisis de asimetría y curtosis muestra que las cinco variables presentan distribuciones alejadas de la normalidad. Se observa una asimetría positiva pronunciada, especialmente en precio y superficies, lo que indica la presencia de valores extremos hacia la derecha.

La curtosis también presenta valores elevados, principalmente en las variables de precio y superficie, lo que refuerza la necesidad de evaluar la influencia de los valores extremos antes de calcular e interpretar las correlaciones.

Estos resultados serán considerados al comparar los coeficientes de Pearson, Spearman y Kendall.

### 10.4. Percentiles adicionales.

In [ ]:
# Cálculo de percentiles adicionales

percentiles = [0.01, 0.05, 0.10, 0.25, 0.50,
               0.75, 0.90, 0.95, 0.99]

display(
    df_analisis[variables_numericas]
    .quantile(percentiles)
    .round(2)
)

## 11. Análisis visual y supuestos

En esta sección se identificarán valores atípicos (outliers) en las principales variables numéricas mediante el método del rango intercuartílico (IQR).

Se calcularán los límites inferior y superior para cada variable y se cuantificará la cantidad y el porcentaje de registros que quedan fuera de esos límites.

Los valores identificados se conservarán para su posterior análisis, sin eliminarlos automáticamente.

### 11.1. Histogramas de las variables numéricas

In [ ]:

# Histogramas de las variables numéricas

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, variable in enumerate(variables_numericas):
    datos = df_analisis[variable].dropna().sample(
        n=min(10000, df_analisis[variable].notna().sum()),
        random_state=42
    )

    sns.histplot(datos, bins=40, kde=True, ax=axes[i])
    axes[i].set_title(f"Distribución de {variable}")
    axes[i].set_xlabel(variable)
    axes[i].set_ylabel("Frecuencia")

# Ocultar el último gráfico vacío
axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

La visualización de los histogramas permite observar que las cinco variables presentan distribuciones asimétricas, con concentración de registros en los valores inferiores y colas hacia la derecha.

Las variables precio y superficies muestran una marcada influencia de valores extremos, mientras que ambientes y antigüedad presentan distribuciones también sesgadas, aunque con características diferentes.

Estos resultados coinciden con las medidas de asimetría y curtosis y con la evaluación de normalidad realizada anteriormente.

### 11.2. Evaluación de normalidad

In [ ]:

# Evaluación de normalidad mediante D'Agostino-Pearson

for variable in variables_numericas:
    datos = df_analisis[variable].dropna().sample(
        n=min(5000, df_analisis[variable].notna().sum()),
        random_state=42
    )

    estadistico, p_valor = normaltest(datos)

    print(f"\nVariable: {variable}")
    print(f"Estadístico: {estadistico:.4f}")
    print(f"Valor p: {p_valor:.6f}")

    if p_valor < 0.05:
        print("Resultado: se rechaza la hipótesis de normalidad.")
    else:
        print("Resultado: no se rechaza la hipótesis de normalidad.")

La evaluación de normalidad mediante el test de D’Agostino-Pearson permitió rechazar la hipótesis nula de normalidad para las cinco variables analizadas.

Estos resultados coinciden con las medidas de asimetría y curtosis calculadas anteriormente, que muestran distribuciones alejadas de la normalidad y presencia de valores extremos.

Se debe tener en cuenta que, debido al tamaño de la muestra, el test puede detectar incluso desviaciones pequeñas respecto de una distribución normal. Por eso, sus resultados se complementan con el análisis de los histogramas y las medidas descriptivas.

11.3. Identificación de distribuciones sesgadas

| Variable | Asimetría | Tipo de distribución |
|---|---:|---|
| p_dolar | 268,09 | Fuertemente sesgada a la derecha |
| sup_tot | 172,20 | Fuertemente sesgada a la derecha |
| sup_cub | 219,22 | Fuertemente sesgada a la derecha |
| amb | 2,61 | Sesgada a la derecha |
| antig | 10,84 | Fuertemente sesgada a la derecha |

El análisis de los histogramas y los coeficientes de asimetría permite identificar que todas las variables presentan distribuciones sesgadas hacia la derecha.

Las variables 'p_dolar', 'sup_tot' y 'sup_cub' muestran una asimetría especialmente pronunciada, debido a la concentración de observaciones en valores bajos y la presencia de registros con valores muy elevados.

En 'amb' se observa una concentración de propiedades entre 1 y 5 ambientes, mientras que 'antig' presenta una distribución concentrada en valores bajos y medios, con algunos registros de antigüedad muy elevada.

Estas características indican que las distribuciones no son simétricas y que los valores extremos pueden influir en el análisis de correlación.

### 11.4. Sensibilidad de Pearson frente a valores extremos

El coeficiente de Pearson mide relaciones lineales y es sensible a la presencia de valores extremos, ya que estos pueden modificar considerablemente su resultado.

En este análisis, las variables presentan distribuciones asimétricas y valores extremos, especialmente el precio y las superficies. Por este motivo, una correlación calculada mediante Pearson podría verse influenciada por estas observaciones.

Para obtener una visión más completa de las relaciones entre las variables, se compararán los resultados de Pearson con los coeficientes no paramétricos de Spearman y Kendall.

## 12. Análisis de valores atípicos (outliers)

En esta sección se representarán gráficamente las distribuciones de las principales variables y las características del mercado inmobiliario. Se utilizarán gráficos Boxplot, Barplot y Violin Plot para facilitar la interpretación de los datos y observar posibles valores extremos.

### 12.1.Detección mediante IQR

In [ ]:
# Detección de valores atípicos mediante el método IQR

resultados_iqr = []

for variable in variables_numericas:

    datos = df_analisis[variable].dropna()

    Q1 = datos.quantile(0.25)
    Q3 = datos.quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = (
        (datos < limite_inferior) |
        (datos > limite_superior)
    )

    resultados_iqr.append({
        "Variable": variable,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Límite inferior": limite_inferior,
        "Límite superior": limite_superior,
        "Cantidad de outliers": outliers.sum(),
        "Porcentaje de outliers": outliers.mean() * 100
    })

tabla_iqr = pd.DataFrame(resultados_iqr)

display(tabla_iqr.round(2))

La aplicación del método IQR permitió identificar posibles valores atípicos en las cinco variables numéricas analizadas.

El precio en dólares presentó la mayor proporción de valores extremos (9,61 %), seguido por las superficies total y cubierta (7,58 % y 7,64 %, respectivamente).

En las variables ambientes y antigüedad, la proporción detectada fue considerablemente menor.

Estos resultados muestran que la presencia de valores extremos tiene distinta intensidad según la variable. Sin embargo, los registros identificados no se consideran necesariamente errores, por lo que se conservarán para evaluar posteriormente su influencia sobre los coeficientes de correlación.

### 12.2. Identificación de valores extremos

In [94]:
# Inspección de los 20 precios más elevados

display(
    df_analisis[
        ["id", "p_dolar", "sup_tot", "sup_cub", "amb", "antig", "barrio"]
    ]
    .nlargest(20, "p_dolar")
)

,id,p_dolar,sup_tot,sup_cub,amb,antig,barrio
106625,42313493,1.111111e+11,64.0,54.0,2.0,NaN,las cañitas
174825,43791821,1.111111e+11,400.0,400.0,5.0,NaN,palermo
192003,43791821,1.111111e+11,400.0,400.0,5.0,NaN,palermo
201635,43791821,1.111111e+11,400.0,400.0,5.0,NaN,palermo
226363,43791821,1.111111e+11,400.0,400.0,5.0,NaN,palermo
236034,43791821,1.111111e+11,400.0,400.0,5.0,NaN,palermo
274856,43791821,1.111111e+11,400.0,400.0,5.0,NaN,palermo
348961,44711996,1.123123e+10,12.0,12.0,12.0,NaN,abasto
361974,44711996,1.123123e+10,12.0,12.0,12.0,NaN,abasto
307571,44440923,1.234568e+09,NaN,NaN,2.0,NaN,saavedra


l revisar las 20 publicaciones con los precios más elevados, se encontraron valores muy superiores a los habituales. También se observan superficies totales extremadamente grandes y varios registros con características similares o repetidas.

Estos casos podrían corresponder a propiedades de características especiales o a posibles inconsistencias en la carga de datos. Como no se cuenta con información suficiente para confirmar su validez, se conservarán para evaluar posteriormente su influencia en las correlaciones.

In [95]:
display(
    df_analisis[
        ["id", "sup_tot", "sup_cub", "p_dolar", "amb", "antig", "barrio"]
    ]
    .nlargest(20, "sup_tot")
    .reset_index(names="indice_original")
)

,indice_original,id,sup_tot,sup_cub,p_dolar,amb,antig,barrio
0,505613,45780358,382637.0,170.00,600000.0,5.0,90.0,liniers
1,81150,43236529,263960.0,263.96,920000.0,5.0,35.0,belgrano
2,101957,43236529,263960.0,263.96,905000.0,5.0,35.0,belgrano
3,115763,43236529,263960.0,263.96,905000.0,5.0,35.0,belgrano
4,127826,43236529,263960.0,263.96,900000.0,5.0,35.0,belgrano
5,132133,43236529,263960.0,263.96,900000.0,5.0,35.0,belgrano
6,139423,43236529,263960.0,263.96,900000.0,5.0,35.0,belgrano
7,151845,43236529,263960.0,263.96,900000.0,5.0,35.0,belgrano
8,396647,44874318,255760.0,199.00,290000.0,3.0,49.0,villa del parque
9,68019,43178986,178800.0,178.80,330000.0,5.0,40.0,flores


Al revisar las 20 publicaciones con mayor superficie total, se encontraron valores muy superiores a las dimensiones habituales de las propiedades.

También se observan diferencias importantes entre la superficie total y la cubierta, además de registros con características similares o repetidas, como ocurre en algunas publicaciones de Belgrano y Caballito.

Estos casos podrían corresponder a propiedades de grandes dimensiones o a posibles inconsistencias en la carga de datos. Como no se cuenta con información suficiente para confirmar su validez, se conservarán para evaluar posteriormente su influencia en las correlaciones.

### 12.3. Preparación para comparar correlaciones con y sin valores atípicos

In [ ]:
# Crear una copia para el análisis sin valores atípicos

df_sin_outliers = df_analisis.copy()

# Identificar registros que no presentan outliers
# en ninguna de las cinco variables numéricas

mascara = pd.Series(True, index=df_analisis.index)

for variable in variables_numericas:

    limites = tabla_iqr[tabla_iqr["Variable"] == variable].iloc[0]

    limite_inferior = limites["Límite inferior"]
    limite_superior = limites["Límite superior"]

    mascara &= (
        df_analisis[variable].between(
            limite_inferior,
            limite_superior
        )
        | df_analisis[variable].isna()
    )

df_sin_outliers = df_analisis.loc[mascara].copy()

print("Registros originales:", len(df_analisis))
print("Registros sin outliers:", len(df_sin_outliers))
print("Registros excluidos:", len(df_analisis) - len(df_sin_outliers))
print("Porcentaje conservado:", round(len(df_sin_outliers) / len(df_analisis) * 100, 2), "%")


Para evaluar la influencia de los valores atípicos en las correlaciones, se creó una copia del conjunto de datos excluyendo los registros que presentaban valores fuera de los límites establecidos por el método IQR en al menos una de las cinco variables numéricas.

De los 519.792 registros originales, se conservaron 455.892 (87,71 %) y se excluyeron 63.900 (12,29 %).

Los valores faltantes no provocaron la exclusión de registros por sí solos. El conjunto original se mantuvo sin modificaciones para permitir la comparación posterior.

## 13. Correlaciones

### 13.1. Matriz de correlación de Pearson



Se calculará la matriz de correlación de Pearson para analizar la intensidad y dirección de las relaciones lineales entre las variables numéricas. Este coeficiente permite identificar si dos variables tienden a aumentar o disminuir conjuntamente.

Debido a que las variables presentan distribuciones asimétricas y valores extremos, los resultados se interpretarán considerando las limitaciones de Pearson.

In [ ]:
# Matriz de correlación de Pearson

correlacion_pearson = df_analisis[variables_numericas].corr(
    method="pearson"
)

display(correlacion_pearson.round(3))

La matriz de Pearson muestra que la mayoría de las relaciones lineales entre las variables analizadas son débiles o prácticamente nulas.

La relación más destacada se observa entre la superficie total y la superficie cubierta (0,377), con una asociación positiva moderada. También se identifica una relación positiva débil entre ambientes y antigüedad (0,279).

Por otra parte, el precio en dólares presenta coeficientes cercanos a cero respecto de todas las características del inmueble. Esto indica que, mediante Pearson, no se identifican relaciones lineales relevantes entre el precio y las variables analizadas.

Estos resultados deben interpretarse considerando la asimetría de las distribuciones y la presencia de valores extremos, que pueden influir en el coeficiente de Pearson.

### 13.2. Matriz de correlación de Spearman.

Se calculará la matriz de correlación de Spearman para analizar la intensidad y dirección de las relaciones monotónicas entre las variables numéricas.

A diferencia de Pearson, Spearman trabaja con rangos en lugar de los valores originales y no requiere una distribución normal. Esto permite evaluar si existen relaciones crecientes o decrecientes, aunque no sean necesariamente lineales.

In [ ]:
# Matriz de correlación de Spearman

correlacion_spearman = df_analisis[variables_numericas].corr(
    method="spearman"
)

display(correlacion_spearman.round(3))

La matriz de Spearman permite identificar asociaciones monotónicas que no se observaban claramente mediante Pearson.

Se destacan las relaciones positivas fuertes entre el precio en dólares y la superficie total (0,849), la superficie cubierta (0,846) y la cantidad de ambientes (0,724).

También se observa una asociación monotónica muy fuerte entre la superficie total y la superficie cubierta (0,957).

La diferencia entre los resultados de Pearson y Spearman sugiere que las relaciones entre el precio y las características de los inmuebles no quedan adecuadamente representadas mediante una correlación lineal simple. La presencia de distribuciones asimétricas y valores extremos podría influir en esta diferencia.

Por lo tanto, Spearman aporta información adicional al análisis, al permitir identificar relaciones monotónicas sin exigir que sean lineales.

Estos resultados representan asociaciones estadísticas y no implican relaciones de causalidad.

### 13.3. Matriz de correlación de Kendall.

Se calculará la matriz de correlación de Kendall para analizar la asociación monotónica entre las variables numéricas, mediante la comparación de pares de observaciones concordantes y discordantes.

Este coeficiente permite evaluar si el orden de los valores de dos variables tiende a coincidir o invertirse. Al igual que Spearman, no requiere normalidad

In [ ]:
# Matriz de correlación de Kendall

correlacion_kendall = df_analisis[variables_numericas].corr(
    method="kendall"
)

display(correlacion_kendall.round(3))

La matriz de Kendall muestra resultados similares a los obtenidos mediante Spearman, identificando asociaciones monotónicas positivas entre el precio y las características de los inmuebles.

Se destacan las relaciones entre el precio en dólares y la superficie total (0,673), la superficie cubierta (0,671) y la cantidad de ambientes (0,578).

La asociación más elevada se observa entre la superficie total y la superficie cubierta (0,865). También se identifican relaciones positivas entre las superficies y la cantidad de ambientes.

Por otra parte, la antigüedad presenta una asociación prácticamente nula con el precio (0,012).

En términos generales, Kendall confirma el patrón observado con Spearman. Aunque sus coeficientes presentan valores numéricos menores, ambos métodos identifican asociaciones monotónicas que Pearson no refleja mediante relaciones lineales.

## 14. Comparación

In [ ]:
# Comparación de los tres métodos de correlación

indices = np.triu_indices(len(variables_numericas), k=1)

comparacion_correlaciones = pd.DataFrame({
    "Variable 1": np.array(variables_numericas)[indices[0]],
    "Variable 2": np.array(variables_numericas)[indices[1]],
    "Pearson": correlacion_pearson.values[indices],
    "Spearman": correlacion_spearman.values[indices],
    "Kendall": correlacion_kendall.values[indices]
})

display(comparacion_correlaciones.round(3))

### 14.1. Comparación general de los tres métodos.

Pearson presenta correlaciones prácticamente nulas entre el precio en dólares y las variables superficie total, superficie cubierta y cantidad de ambientes. En cambio, Spearman y Kendall identifican asociaciones monotónicas positivas, especialmente con las superficies y la cantidad de ambientes.

La relación entre superficie total y superficie cubierta también muestra diferencias en su intensidad: Pearson obtiene 0,377, mientras que Spearman alcanza 0,957 y Kendall 0,865.

En términos generales, Spearman y Kendall permiten identificar patrones de asociación que Pearson no refleja adecuadamente mediante relaciones lineales. Esto puede estar relacionado con la asimetría de las distribuciones y la presencia de valores extremos observados durante el análisis exploratorio.

Por otra parte, la antigüedad presenta asociaciones débiles o prácticamente nulas con el precio en los tres métodos.

Estos resultados muestran la importancia de utilizar diferentes coeficientes para obtener una visión más completa de las relaciones entre las variables. Las asociaciones identificadas no implican causalidad.

### 14.2. Comparación de correlaciones con y sin valores atípicos.

Se calcularán nuevamente los coeficientes de Pearson, Spearman y Kendall utilizando el conjunto de datos sin valores atípicos. Luego, se compararán estos resultados con las matrices originales para identificar posibles cambios en la intensidad y dirección de las asociaciones.

In [ ]:
correlacion_pearson_sin_outliers = df_sin_outliers[variables_numericas].corr(
    method="pearson"
)

correlacion_spearman_sin_outliers = df_sin_outliers[variables_numericas].corr(
    method="spearman"
)

correlacion_kendall_sin_outliers = df_sin_outliers[variables_numericas].corr(
    method="kendall"
)

print("Pearson sin valores atípicos:")
display(correlacion_pearson_sin_outliers.round(3))

print("Spearman sin valores atípicos:")
display(correlacion_spearman_sin_outliers.round(3))

print("Kendall sin valores atípicos:")
display(correlacion_kendall_sin_outliers.round(3))

La comparación de las matrices con y sin valores atípicos permite observar diferencias importantes en los resultados de los tres métodos.

Pearson presenta los cambios más pronunciados. En la matriz original, las correlaciones entre el precio en dólares y las superficies eran prácticamente nulas. Sin embargo, al excluir los valores atípicos, estas asociaciones pasan a ser positivas y fuertes, alcanzando 0,757 para la superficie total y 0,766 para la superficie cubierta. También aumenta la correlación con la cantidad de ambientes, de 0,006 a 0,611.

En cambio, Spearman y Kendall presentan variaciones más moderadas. Ambos mantienen asociaciones positivas entre el precio, las superficies y la cantidad de ambientes, aunque sus coeficientes disminuyen ligeramente después de excluir los valores atípicos.

La relación entre superficie total y superficie cubierta también aumenta considerablemente en Pearson, pasando de 0,377 a 0,924. En Spearman y Kendall, en cambio, se observan disminuciones leves.

En términos generales, estos resultados muestran que Pearson fue el método más afectado por la exclusión de los valores atípicos. Esto evidencia la sensibilidad de este coeficiente frente a observaciones extremas y cómo pueden influir en la medición de relaciones lineales.

### 14.3. Identificación de relaciones monotónicas no lineales.

Para identificar posibles relaciones monotónicas no lineales, se comparan los resultados de Pearson con los de Spearman y Kendall.

Cuando Spearman y Kendall muestran asociaciones positivas o negativas más fuertes que Pearson, puede existir una relación monotónica que no se ajusta adecuadamente a un patrón lineal.

En este análisis se observan diferencias especialmente importantes entre el precio en dólares y las superficies, así como entre el precio y la cantidad de ambientes.

### Relación entre precio y características de los inmuebles (escala original)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

variables_comparar = ["sup_tot", "sup_cub", "amb"]

for i, variable in enumerate(variables_comparar):
    sns.scatterplot(
        data=df_analisis.sample(n=10000, random_state=42),
        x=variable,
        y="p_dolar",
        alpha=0.3,
        ax=axes[i]
    )
    axes[i].set_title(f"Precio en dólares vs. {variable}")
    axes[i].set_xlabel(variable)
    axes[i].set_ylabel("Precio en dólares")

plt.tight_layout()
plt.show()

escala original, este gráfico permite mostrar el efecto de los valores extremos y la concentración de los registros.

### Relación entre precio y características de los inmuebles (escala logarítmica)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

muestra = df_analisis.sample(n=10000, random_state=42)

for i, variable in enumerate(["sup_tot", "sup_cub", "amb"]):
    sns.scatterplot(
        data=muestra,
        x=variable,
        y="p_dolar",
        alpha=0.3,
        ax=axes[i]
    )
    axes[i].set_yscale("log")
    axes[i].set_title(f"Precio vs. {variable}")
    axes[i].set_xlabel(variable)
    axes[i].set_ylabel("Precio en dólares (escala logarítmica)")

plt.tight_layout()
plt.show()

Escala logarítmica, este gráfico permite observar mejor la distribución de los precios y las tendencias que quedan ocultas en el primero.

### 14.4. Selección del coeficiente más adecuado para estos datos.

En este conjunto de datos, las variables presentan distribuciones asimétricas, valores extremos y diferencias importantes entre los resultados de Pearson, Spearman y Kendall.

Pearson mostró una elevada sensibilidad a la exclusión de valores atípicos, mientras que Spearman y Kendall conservaron patrones de asociación más estables.

Por este motivo, Spearman resulta el coeficiente más adecuado como medida principal para analizar las relaciones entre las variables, ya que permite identificar asociaciones monotónicas sin exigir relaciones lineales ni distribuciones normales. Kendall puede utilizarse como medida complementaria para contrastar los resultados.
¿Por qué elegimos Spearman?
- No requiere normalidad de las variables.
- Permite identificar relaciones monotónicas, aunque no sean lineales.
- Trabaja con rangos y no directamente con los valores originales.
- En nuestros resultados mostró patrones claros de asociación entre precio, superficies y ambientes.
Importante: no significa que Spearman sea inmune a los valores atípicos. Su comportamiento también puede verse afectado por observaciones extremas, aunque suele ser menos sensible que Pearson.

| Criterio | Resultado observado | Método más adecuado |
|---|---|---|
| Distribuciones no normales | Las cinco variables no presentan normalidad | Spearman / Kendall |
| Relaciones monotónicas | Spearman y Kendall identifican asociaciones positivas | Spearman / Kendall |
| Sensibilidad a valores extremos | Pearson presenta los mayores cambios | Spearman / Kendall |
| Relaciones lineales | Pearson mide específicamente asociaciones lineales | Pearson |

## 15. Conclusiones


### 15.1. Interpretación de la relación entre el precio y las características de los inmuebles

Los resultados muestran que existe una asociación positiva entre el precio en dólares y las características estructurales de las propiedades, especialmente la superficie total, la superficie cubierta y la cantidad de ambientes.

Estas relaciones se identifican principalmente mediante Spearman y Kendall, que muestran que, en general, los inmuebles de mayor superficie y con más ambientes tienden a presentar precios más elevados.

En cambio, la antigüedad presenta asociaciones débiles o prácticamente nulas con el precio, por lo que no se observa una relación monotónica clara entre ambas variables en este conjunto de datos.

### 15.2. Justificación del coeficiente elegido

Se seleccionó Spearman como coeficiente principal porque las variables presentan distribuciones asimétricas, valores extremos y no cumplen con el supuesto de normalidad.

Además, permitió identificar asociaciones monotónicas que Pearson no reflejó adecuadamente mediante relaciones lineales. Kendall se utilizó como medida complementaria para contrastar los resultados.

La comparación con y sin valores atípicos también permitió observar la sensibilidad de Pearson frente a la exclusión de registros extremos.

### 15.3. Correlación no implica causalidad

Las asociaciones identificadas no permiten afirmar que una característica de la propiedad provoque un aumento o una disminución del precio.

Por ejemplo, aunque se observa una asociación positiva entre la superficie y el precio, esto no significa que aumentar la superficie sea la única causa de un mayor valor de venta. Existen otros factores que pueden intervenir, como la ubicación, el estado de conservación, la calidad constructiva y las condiciones del mercado inmobiliario.

### 15.4. Limitaciones del análisis

Durante el análisis se identificaron algunas limitaciones:
- Presencia de valores extremos que pueden influir en los coeficientes de correlación.
- Distribuciones asimétricas y ausencia de normalidad en las variables analizadas.
- Existencia de valores faltantes, que pueden reducir la cantidad de observaciones utilizadas en cada correlación.
- Algunos registros presentan valores inusuales cuya validez no pudo confirmarse.
- El análisis se concentra en variables numéricas y no incorpora otras características que podrían influir en el precio, como la ubicación o el estado del inmueble.